**Aceleradores de Hardware - Ordem de Custo**

CPU

GPUs: T4

GPUs: L4

TPU v5e-1

TPU v6e-1

GPUs: A100

GPUs: G4

GPUs: H100

### Import Libraries and Set the intial variables

In [ ]:
#!pip install transformers

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pylab import rcParams

from sklearn.metrics import confusion_matrix, classification_report

# Torch ML libraries
from transformers import BertModel, AutoTokenizer, get_linear_schedule_with_warmup
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader

# Misc.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set intial variables and constants
%config InlineBackend.figure_format='retina'

# Graph Designs
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
HAPPY_COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(HAPPY_COLORS_PALETTE))
rcParams['figure.figsize'] = 12, 8

# Random seed for reproducibilty
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Set GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


##Parametrização conforme ambiente de execução.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_PATH_GENERAL = '/content/drive/MyDrive/_Mestrado PUCPR/'
BASE_PATH_DATASET = f'{BASE_PATH_GENERAL}Datasets/'

LISTA_MODEL_CHECKPOINT = [
    {
        'endpoint': 'raquelsilveira/legalbertpt_fp',
        'descricao': 'LegalBERTpt Jurídico',
        'num_layers': 12
    },
    {
        'endpoint': 'rufimelo/Legal-BERTimbau-base',
        'descricao': 'Legal-BERTimbau Jurídico Base',
        'num_layers': 12
    },
    {
        'endpoint': 'rufimelo/Legal-BERTimbau-large',
        'descricao': 'Legal-BERTimbau Jurídico Large',
        'num_layers': 24
    },
    {
        'endpoint': 'neuralmind/bert-base-portuguese-cased',
        'descricao': 'BERTimbau Base',
        'num_layers': 12
    },
    {
        'endpoint': 'neuralmind/bert-large-portuguese-cased',
        'descricao': 'BERTimbau Large',
        'num_layers': 24
    },
    {
        'endpoint': 'google-bert/bert-base-multilingual-cased',
        'descricao': 'BERT Multilingual Base',
        'num_layers': 12
    },
]

RANDOM_STATE = 42
EPOCHS = 2
APPROACH_EXECUTION = 'epoch' #'step'
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING = 5
class_names = ['Desfavorável ao Réu', 'Favorável ao Réu']
MAX_LEN = 512

# Performance knobs
NUM_WORKERS = 2

In [ ]:
import os

# Paths padrão para reprodutibilidade
train_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2026_train-punct.csv"
test_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2026_test-punct.csv"

# Apenas carrega os arquivos já persistidos (sem split)
if not os.path.exists(train_path):
    raise FileNotFoundError(f"Arquivo de treino não encontrado: {train_path}")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Arquivo de teste não encontrado: {test_path}")

print("Carregando arquivos de treino e teste persistidos...")
df_train = pd.read_csv(train_path, sep=';')
df_test = pd.read_csv(test_path, sep=';')

print(f"Treino carregado de: {train_path} | shape: {df_train.shape}")
print(f"Teste carregado de: {test_path} | shape: {df_test.shape}")

Arquivos de treino e teste encontrados. Reutilizando arquivos persistidos...
Treino carregado de: /content/drive/MyDrive/_Mestrado PUCPR/Datasets/preprocessed_acordaos_HC_under_2021_2026_treino.csv | shape: (4976, 22)
Teste carregado de: /content/drive/MyDrive/_Mestrado PUCPR/Datasets/preprocessed_acordaos_HC_under_2021_2026_teste.csv | shape: (1244, 22)
Dataset combinado (df) | shape: (6220, 22)


In [ ]:
# Function para converter a decisão em valor inteiro (0 = Desfavorável, 1 = Favorável)
def cast_decisao_to_int(decisao_favoravel):
    if decisao_favoravel == 'S':
        return 1
    else:
        return 0

# Apply to train and test datasets
if 'DECISAO_FAVORAVEL' in df_train.columns and 'classe' not in df_train.columns:
    df_train['classe'] = df_train.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)
if 'DECISAO_FAVORAVEL' in df_test.columns and 'classe' not in df_test.columns:
    df_test['classe'] = df_test.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)

print(f"Classes no treino: {df_train['classe'].value_counts().to_dict()}")
print(f"Classes no teste:  {df_test['classe'].value_counts().to_dict()}")

# BERT

### Preparing Torch Dataset

To enter data into a PyTorch, we need a more robust data generator class. We will return the review text as well to validate our predictions easily.

###  `HCAcordaoSTJ` (Custom Dataset)

A classe `HCAcordaoSTJ` (que herda de `torch.utils.data.Dataset`) atua como uma **ponte essencial entre os seus dados brutos e o modelo PyTorch/HuggingFace**. A criação desta classe customizada é uma prática padrão e necessária pelos seguintes motivos:

1. **Contrato com o PyTorch (`DataLoader`)**: Para que o `DataLoader` consiga gerenciar os lotes (batches), embaralhar os dados e usar múltiplos processadores, ele precisa saber como interagir com seus dados. O PyTorch exige que você implemente um contrato básico usando dois métodos:
   * `__len__`: Informa ao DataLoader o tamanho total do dataset.
   * `__getitem__`: Ensina o DataLoader a extrair e processar uma única linha (amostra) com base em um índice.

2. **Tokenização *On-the-Fly* (Sob Demanda) e Eficiência de Memória**: O método `__getitem__` aplica o `tokenizer` do HuggingFace individualmente em cada texto *apenas no momento em que a amostra é chamada* durante o treinamento. Em vez de tokenizar todo o seu dataset de 6336 acórdãos de uma vez e armazenar matrizes gigantescas na memória RAM, a transformação é feita lote a lote.

3. **Formatação Exigida pelo BERT**: Modelos da biblioteca `transformers` do HuggingFace não aceitam texto puro (strings). Eles esperam tensores (matrizes numéricas do PyTorch) com chaves muito específicas. O método `__getitem__` prepara e empacota exatamente o que o BERT precisa:
   * **`input_ids`**: O texto mapeado para os números do vocabulário do modelo.
   * **`attention_mask`**: Uma máscara de 1s e 0s que ensina o modelo a olhar apenas para as palavras reais e ignorar os espaços vazios (*padding*).

4. **Padronização Universal (*Padding* e *Truncation*)**: Redes neurais precisam que todas as entradas de um lote tenham o mesmo tamanho matemático. Como os acórdãos variam de tamanho, a classe força com que todos tenham exatamente `max_len` (no seu caso, 512 tokens). Textos menores são preenchidos com zeros (`padding='max_length'`) e textos maiores são cortados (`truncation=True`).

5. **Empacotamento dos Alvos (*Targets*)**: Além de formatar a entrada para o HuggingFace, a classe converte as suas classes (0 ou 1) no formato `torch.long`, que é o tipo de dado estritamente exigido pela função de perda (`CrossEntropyLoss`) do PyTorch durante a etapa de otimização.

Em resumo, o `HCAcordaoSTJ` é o "tradutor" que pega suas strings do Pandas e as transforma perfeitamente no idioma matemático e estrutural que o BERT e o PyTorch entendem.

In [ ]:
from typing import Dict, Union
import numpy as np
import torch
from torch.utils.data import Dataset

class HCAcordaoSTJ(Dataset):
    def __init__(self, reports: np.ndarray, targets: np.ndarray, tokenizer, max_len: int) -> Dataset:
        self.reports = reports.astype(str)
        self.targets = targets

        # Tokeniza em lote uma vez para reduzir custo por época
        encodings = tokenizer(
            self.reports.tolist(),
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True
        )

        self.input_ids = torch.tensor(encodings['input_ids'], dtype=torch.long)
        self.attention_mask = torch.tensor(encodings['attention_mask'], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.reports)

    def __getitem__(self, item: int) -> Dict[str, Union[str, torch.Tensor]]:
        return {
            'report_text': self.reports[item],
            'input_ids': self.input_ids[item],
            'attention_mask': self.attention_mask[item],
            'targets': torch.tensor(self.targets[item], dtype=torch.long)
        }

### Otimização de Memória
Manter apenas as colunas estritamente necessárias para o treinamento do modelo.

In [ ]:
import gc

# Criar um novo DataFrame apenas com as colunas necessárias
df_clean = df[['RELATORIO', 'classe']].copy()

# Excluir o dataframe antigo da memória e forçar a coleta de lixo
del df
gc.collect()

# Renomear de volta para 'df' para manter a compatibilidade com o código abaixo
df = df_clean

print("Memória otimizada. Colunas atuais:", df.columns.tolist())
print("Novo shape do DataFrame:", df.shape)
display(df.head(3))

Memória otimizada. Colunas atuais: ['RELATORIO', 'classe']
Novo shape do DataFrame: (6220, 2)


,RELATORIO,classe
0,Trata-se de agravo regimental interposto pelo ...,0
1,Trata-se de agravo regimental interposto por E...,0
2,O EXMO. SR. MINISTRO ANTONIO SALDANHA PALHEIRO...,0


Create a 80% train data and 20% test.

In [ ]:
# Split já é tratado na célula de carga inteligente (Célula 10).
# Aqui apenas conferimos os dataframes finais para manter compatibilidade do fluxo.
if 'df_train' not in globals() or 'df_test' not in globals():
    raise RuntimeError("Execute primeiro a célula de carga inteligente para criar/carregar treino e teste.")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

Train shape: (4976, 22)
Test shape: (1244, 22)


In [ ]:
#Você parou nesta etapa. Agora vc precisa estudar em detalhes a classe DataLoader e as implicações com o treinamento.

Create a dataloader to release data in batches.

### O papel do DataLoader neste treinamento

O `DataLoader` em PyTorch é uma utilidade fundamental para o treinamento de modelos de aprendizado profundo, especialmente com grandes conjuntos de dados. No contexto deste treinamento BERT, ele desempenha vários papéis cruciais:

1.  **Iteração em Lotes (Batching)**: Ele agrupa os dados em pequenos lotes (definidos por `batch_size`). Em vez de alimentar uma única amostra para o modelo por vez (o que é ineficiente para GPUs), o `DataLoader` fornece um lote de amostras. Isso permite que o modelo processe várias entradas simultaneamente, acelerando o treinamento e aproveitando a paralelização do hardware.

2.  **Embaralhamento (Shuffling)**: Antes de cada época de treinamento, o `DataLoader` (se configurado com `shuffle=True`, embora neste notebook esteja implícito que não está explícito no `create_data_loader` mas é uma prática comum para o treinamento) embaralha os dados. Isso é vital para garantir que o modelo não aprenda a ordem dos dados e para evitar que a ordem das amostras introduza vieses no treinamento.

3.  **Processamento Multiprocessado (`num_workers`)**: O parâmetro `num_workers` permite que os dados sejam carregados e pré-processados em paralelo por múltiplos processos secundários (em vez do processo principal). Isso é particularmente útil quando o pré-processamento de dados é uma etapa demorada (como a tokenização e o padding feitos na classe `HCAcordaoSTJ`), pois evita gargalos no carregamento de dados que poderiam deixar a GPU ociosa.

4.  **Integração com `Dataset`**: O `DataLoader` trabalha em conjunto com a classe `Dataset` (neste caso, `HCAcordaoSTJ`). A classe `Dataset` é responsável por como as amostras individuais são carregadas e pré-processadas (tokenização, truncamento, padding, etc.), enquanto o `DataLoader` é responsável por como essas amostras individuais são agrupadas e fornecidas ao modelo em lotes.

Em resumo, o `DataLoader` otimiza o fluxo de dados do disco para a memória e, em seguida, para o modelo, garantindo que o treinamento seja eficiente, reprodutível (com `random_state`) e que o modelo veja os dados de forma variada ao longo das épocas, o que é fundamental para um bom desempenho e generalização.

In [ ]:
import os

def create_data_loader(df: pd.DataFrame, tokenizer, max_len: int, batch_size: int, shuffle: bool = False) -> DataLoader:
    ds = HCAcordaoSTJ(
        reports=df.RELATORIO.to_numpy(),
        targets=df.classe.to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )

    workers = min(NUM_WORKERS, os.cpu_count() or 1)

    loader_kwargs = {
        'dataset': ds,
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': workers,
        'pin_memory': (device.type == 'cuda')
    }

    if workers > 0:
        loader_kwargs['persistent_workers'] = True
        loader_kwargs['prefetch_factor'] = 2

    return DataLoader(**loader_kwargs)

## Classificação de Decisão com BERT e Hugging Face

Usaremos o BertModel base e construiremos um classificador para prever se a decisão será favorável ou desfavorável ao réu.

##### Load the basic BERT model
bert_model = BertModel.from_pretrained(MODEL_NAME)

We use a dropout layer for some regularization and a fully-connected layer for our output. We are returning the raw output of the last layer since that is required for the cross-entropy loss function in PyTorch to work. Create an instance and move it to the GPU

In [ ]:
# Bloco de testes alternativos de modelo (mantido apenas como referência)
# from transformers import AutoModelForSequenceClassification
# bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

In [ ]:
# num_layers_to_freeze = LAYERS_TO_FREEZE
# print(f"Freezing the embeddings and the first {num_layers_to_freeze} encoder layers of the model...")

# for name, param in bert_model.named_parameters():
#     if 'embeddings' in name:
#         param.requires_grad = False
#     elif 'encoder.layer' in name:
#         # Extrai o número da camada pelo nome (ex: bert.encoder.layer.0.attention...)
#         layer_num = int(name.split('.')[3])
#         if layer_num < num_layers_to_freeze:
#             param.requires_grad = False
#         else:
#             # Garante que camadas subsequentes sejam treináveis
#             param.requires_grad = True
#     else:
#         # O pooler e o classifier (camadas finais) continuam treináveis
#         param.requires_grad = True

# print("Layers freezing complete. You can now proceed with training.")

In [ ]:
# Print layer names
# The index can help you identify which layers correspond to earlier blocks (e.g., embedding, early encoder layers)
# i = 0
# for name, param in bert_model.named_parameters():
#   i = i + 1
#   print(f"index: {i} {name} - status: {param.requires_grad}")

In [ ]:
#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

### O papel da classe `HCAcordaoClassifier` no Fine-Tuning

A classe `HCAcordaoClassifier` é o coração do seu processo de *fine-tuning*. Ela herda de `nn.Module` (a classe base para todas as redes neurais no PyTorch) e tem a função de **adaptar um modelo de linguagem genérico para a sua tarefa específica** de classificação binária (Favorável vs. Desfavorável).

Com as recentes modificações, a arquitetura ficou mais profunda e robusta. Aqui está o que cada parte faz e onde exatamente estão as camadas *fully connected*:

1. **A Base (O "Cérebro" Pré-treinado): `self.bert`**
   * Utilizamos o `BertModel.from_pretrained()` do Hugging Face.
   * O BERT atua como um poderoso **extrator de características**. Ele lê o texto e o transforma em uma rica representação matemática (um vetor de 768 dimensões).

2. **Camadas Intermediárias (Rede Fully Connected / Multi-Layer Perceptron)**
   * Em vez de ligar o BERT direto na saída, criamos uma rede neural intermediária mais profunda com três camadas lineares (`self.fc1`, `self.fc2`, `self.fc3`).
   * **É aqui que começa a sua rede *fully connected* (totalmente conectada)**, construída com `nn.Linear`. O fluxo expande a capacidade de aprendizado projetando as 768 dimensões iniciais para **4000 neurônios**, funilando depois para **2000** e, em seguida, para **1000 neurônios**. Em uma camada *fully connected*, todos os neurônios se conectam a todos os neurônios da camada seguinte, permitindo que o modelo aprenda padrões semânticos jurídicos altamente complexos.

3. **Ativação (`ReLU`) e Regularização (`Dropout`)**
   * **`ReLU`:** Aplicada após cada camada intermediária *fully connected*, é a função matemática não-linear que permite à rede extrair e combinar características de maneira mais complexa.
   * **`Dropout` (p=0.3):** Desliga aleatoriamente 30% dos neurônios durante o treinamento. É absolutamente crucial nesta arquitetura grande (com muitos parâmetros) para combater o *overfitting* (impedir que o modelo apenas "decore" os dados de treino).

4. **A Cabeça de Classificação (A última Fully Connected): `self.out`**
   * A camada `self.out`, definida como `nn.Linear(1000, n_classes)`, é a **última camada *fully connected*** da sua rede. Pega os 1000 neurônios da última camada oculta e os reduz para apenas **2 números** (suas `n_classes`), que representarão as predições brutas (logits) do acórdão ser Favorável ou Desfavorável.

5. **O Fluxo de Dados: Método `forward`**
   * Recebe os `input_ids` e a `attention_mask` (fornecidos pelo `DataLoader`).
   * O BERT processa o texto e fornece o **`pooler_output`** (o "resumo" do texto baseado no token especial `[CLS]`).
   * Esse vetor de resumo passa sequencialmente pelos blocos *fully connected* da seguinte forma: `Camada Linear -> ReLU -> Dropout`.
   * Finalmente, passa por `self.out` para dar o veredito final.

In [ ]:
class HCAcordaoClassifier(nn.Module):

    # Mapa de arquitetura da cabeça classificadora por número de camadas do modelo
    HEAD_CONFIG = {
        12: (256, 128, 64),
        24: (512, 256, 128),
    }

    def __init__(self, n_classes: int, num_layers: int, model_name: str) -> None:
        super(HCAcordaoClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)

        if num_layers not in self.HEAD_CONFIG:
            raise ValueError(f"num_layers inválido ({num_layers}). Valores suportados: {list(self.HEAD_CONFIG.keys())}")

        h1, h2, h3 = self.HEAD_CONFIG[num_layers]

        self.fc1 = nn.Linear(self.bert.config.hidden_size, h1)
        self.drop1 = nn.Dropout(p=0.3)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(h1, h2)
        self.drop2 = nn.Dropout(p=0.3)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(h2, h3)
        self.drop3 = nn.Dropout(p=0.3)
        self.relu3 = nn.ReLU()

        self.out = nn.Linear(h3, n_classes)

        self.freeze_layers(num_layers)

    def freeze_layers(self, num_layers_to_freeze: int) -> None:
        print(f"Freezing embeddings + first {num_layers_to_freeze} encoder layers...")
        for name, param in self.bert.named_parameters():
            if 'embeddings' in name:
                param.requires_grad = False
            elif 'encoder.layer' in name:
                layer_num = int(name.split('.')[2])
                if layer_num < num_layers_to_freeze:
                    param.requires_grad = False
                else:
                    param.requires_grad = True
            else:
                param.requires_grad = True
        print("Layers freezing complete.")

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.fc1(output.pooler_output)
        hidden = self.relu1(hidden)
        hidden = self.drop1(hidden)
        hidden = self.fc2(hidden)
        hidden = self.relu2(hidden)
        hidden = self.drop2(hidden)
        hidden = self.fc3(hidden)
        hidden = self.relu3(hidden)
        hidden = self.drop3(hidden)
        return self.out(hidden)


In [ ]:
%%time
import os, gc, copy
from collections import defaultdict
from torch.optim import AdamW
from sklearn.metrics import precision_score, recall_score, f1_score
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc

# ---------------------------------------------------------------------------
# Funções auxiliares (parametrizadas)
# ---------------------------------------------------------------------------
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples, scaler, use_amp):
    model.train()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []
    optimizer.zero_grad(set_to_none=True)

    for d in data_loader:
        input_ids = d["input_ids"].to(device, non_blocking=True)
        attention_mask = d["attention_mask"].to(device, non_blocking=True)
        targets = d["targets"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs, targets)

        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def eval_model(model, data_loader, loss_fn, device, n_examples, use_amp=False):
    model.eval()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs, targets)

            _, preds = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def get_predictions(model, data_loader, device):
    model.eval()
    predictions, prediction_probs, real_values = [], [], []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds)
            prediction_probs.extend(outputs)
            real_values.extend(targets)
    return (torch.stack(predictions).cpu(),
            torch.stack(prediction_probs).cpu(),
            torch.stack(real_values).cpu())


# ---------------------------------------------------------------------------
# Loop principal: fine-tuning + avaliação para cada modelo
# ---------------------------------------------------------------------------
all_roc_data = []       # (descricao, fpr, tpr, roc_auc)
all_metrics = []        # dict por modelo

model_dir = f'{BASE_PATH_GENERAL}models'
os.makedirs(model_dir, exist_ok=True)

for idx, model_cfg in enumerate(LISTA_MODEL_CHECKPOINT):
    MODEL_CHECKPOINT = model_cfg['endpoint']
    MODEL_DESC = model_cfg['descricao']
    NUM_LAYERS = model_cfg['num_layers']

    print("\n" + "=" * 80)
    print(f"[{idx+1}/{len(LISTA_MODEL_CHECKPOINT)}] {MODEL_DESC} ({MODEL_CHECKPOINT})")
    print("=" * 80)

    # --- Tokenizer ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

    # --- DataLoaders ---
    train_data_loader = create_data_loader(df_train, tokenizer, MAX_LEN, BATCH_SIZE, shuffle=True)
    test_data_loader = create_data_loader(df_test, tokenizer, MAX_LEN, BATCH_SIZE, shuffle=False)

    # --- Modelo ---
    model = HCAcordaoClassifier(len(class_names), NUM_LAYERS, MODEL_CHECKPOINT)
    model = model.to(device)

    # --- Optimizer / Scheduler / Loss ---
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_data_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    loss_fn = nn.CrossEntropyLoss().to(device)
    use_amp = (device.type == 'cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # --- Treinamento ---
    history = defaultdict(list)
    best_recall, best_model_weights = 0, None
    epochs_no_improve = 0

    for epoch in range(EPOCHS):
        print(f"\n  Epoch {epoch+1}/{EPOCHS}")
        print("  " + "-" * 60)

        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_data_loader, loss_fn, optimizer, device, scheduler, len(df_train), scaler, use_amp)
        print(f"  Train | loss {train_loss:.4f} acc {train_acc:.4f} prec {train_prec:.4f} rec {train_rec:.4f} f1 {train_f1:.4f}")

        test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(
            model, test_data_loader, loss_fn, device, len(df_test), use_amp)
        print(f"  Test  | loss {test_loss:.4f} acc {test_acc:.4f} prec {test_prec:.4f} rec {test_rec:.4f} f1 {test_f1:.4f}")

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)

        if test_rec > best_recall:
            best_model_weights = copy.deepcopy(model.state_dict())
            best_recall = test_rec
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING:
                print(f"  Early Stopping na época {epoch+1}.")
                break

    # --- Gráfico de Loss ---
    plt.figure()
    plt.plot(history['train_loss'], label='train loss')
    plt.plot(history['test_loss'], label='val loss')
    plt.title(f'Loss — {MODEL_DESC}')
    plt.ylabel('Loss'); plt.xlabel('Epoch'); plt.legend(); plt.ylim([0, 2])
    plt.show()

    # --- Carregar melhores pesos ---
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        model_save_path = f'{model_dir}/best_model_{MODEL_CHECKPOINT.replace("/","_")}.bin'
        torch.save(best_model_weights, model_save_path)
        print(f"  Melhor modelo salvo em: {model_save_path}")

    # --- Avaliação Final ---
    test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(
        model, test_data_loader, loss_fn, device, len(df_test), use_amp)
    print(f"\n  Métricas finais ({MODEL_DESC}):")
    print(f"  Accuracy: {test_acc:.4f} | Precision: {test_prec:.4f} | Recall: {test_rec:.4f} | F1: {test_f1:.4f}")

    # --- Predictions ---
    y_pred, y_pred_probs, y_test = get_predictions(model, test_data_loader, device)

    # --- Classification Report ---
    print(f"\n  Classification Report — {MODEL_DESC}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # --- Confusion Matrix ---
    cm = confusion_matrix(y_test, y_pred)
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    plt.figure()
    hmap = sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    hmap.yaxis.set_ticklabels(hmap.yaxis.get_ticklabels(), rotation=0, ha='right')
    hmap.xaxis.set_ticklabels(hmap.xaxis.get_ticklabels(), rotation=30, ha='right')
    plt.ylabel('Resultado do Julgamento'); plt.xlabel('Decisão Predita')
    plt.title(f'Matriz de Confusão — {MODEL_DESC}')
    plt.show()

    # --- Curva ROC individual ---
    probs = F.softmax(y_pred_probs, dim=1).numpy()
    y_scores = probs[:, 1]
    fpr, tpr, _ = roc_curve(y_test.numpy(), y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--')
    plt.xlim([0, 1]); plt.ylim([0, 1.05])
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'Curva ROC — {MODEL_DESC}')
    plt.legend(loc="lower right"); plt.grid(alpha=0.3, linestyle='--')
    plt.show()
    print(f"  AUC: {roc_auc:.4f}")

    # --- Armazena para comparação final ---
    all_roc_data.append((MODEL_DESC, fpr, tpr, roc_auc))
    all_metrics.append({
        'Model': MODEL_DESC,
        'Accuracy': round(test_acc, 4),
        'Precision': round(test_prec, 4),
        'Recall': round(test_rec, 4),
        'F1-Score': round(test_f1, 4),
        'AUC': round(roc_auc, 4),
    })

    # --- Liberar memória ---
    del model, optimizer, scheduler, scaler, tokenizer
    del train_data_loader, test_data_loader
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print("\n\n" + "=" * 80)
print("Trade-off concluído para todos os modelos.")
print("=" * 80)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: rufimelo/Legal-BERTimbau-large
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Freezing the embeddings and the first 24 encoder layers of the model...
Layers freezing complete. You can now proceed with training.
1024


## Comparação Final — Curva ROC (Todos os Modelos)

In [ ]:
# --- Curva ROC Comparativa ---
plt.figure(figsize=(10, 8))
colors = plt.cm.get_cmap('tab10', len(all_roc_data))

for i, (desc, fpr, tpr, roc_auc) in enumerate(all_roc_data):
    plt.plot(fpr, tpr, color=colors(i), lw=2, label=f'{desc} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curva ROC Comparativa — Trade-off de Modelos', fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## Tabela Comparativa de Métricas

In [ ]:
# --- Tabela de Métricas Finais ---
df_metrics = pd.DataFrame(all_metrics)
df_metrics = df_metrics.sort_values('AUC', ascending=False).reset_index(drop=True)
print("=" * 80)
print("TABELA COMPARATIVA DE MÉTRICAS — TRADE-OFF DE MODELOS")
print("=" * 80)
display(df_metrics.style.highlight_max(
    subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
    color='lightgreen'
))